# Week 2: Understanding the Problem

## What you will learn this week

- Why statistics **describe** the past but cannot **predict** an individual
- What a **model** is and why we need one
- What **error** means — and how to measure it
- The difference between **supervised learning** and just computing averages
- A first look at **overfitting** — when a model memorizes instead of learning

We work through four real-world scenarios where a statistic is available but is clearly not enough.

---

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

plt.style.use('seaborn-v0_8-whitegrid')
matplotlib.rcParams['figure.dpi'] = 110
matplotlib.rcParams['font.size'] = 11

print('Libraries loaded.')

---

## The Core Idea Before We Start

**Statistics tells you about groups. Models make predictions about individuals.**

| Statistics says... | What we actually need... |
|---|---|
| 70% of tall people are fast | Is *this particular person* fast? |
| Average house price is $350k | What should *this specific house* cost? |
| 80% of emails with FREE are spam | Is *this specific email* spam? |
| Average grade is 72 | What grade will *this student* get? |

A **model** takes multiple features about an individual and combines them into a prediction.

---

## Scenario 1: Loan Approval

### What statistics tells us

A bank has data showing:
- 70% of high-income applicants get approved
- 40% of low-income applicants get approved

### The limitation

A new applicant walks in with high income. Should we approve them?

If we only use the statistic, we say "70% chance — approve!" But this ignores:
- Their credit score (history of repayment)
- Their existing debt
- Their age and employment stability

High income + poor credit history + existing debt → should be rejected.

**We need a model that combines all these signals together.**

In [ ]:
# ── Loan Approval: Statistics vs Model Thinking ───────────────────────────────

# Applicant data: [income_k, credit_score, debt_k, age]
# Actual outcomes: 1=approved, 0=rejected (real bank decisions)
applicants = np.array([
    [90,  750, 10, 35],   # approved
    [85,  680, 25, 28],   # rejected — high debt, lower credit
    [120, 800, 5,  42],   # approved
    [70,  720, 15, 31],   # approved
    [110, 580, 40, 26],   # rejected — poor credit, high debt
    [95,  760, 8,  38],   # approved
])
labels  = np.array([1, 0, 1, 1, 0, 1])
feat_names = ['Income ($k)', 'Credit Score', 'Debt ($k)', 'Age']

print('=== LOAN APPLICATION DATA ===')
print(f'  {"Applicant":12} {"Income":>10} {"Credit":>8} {"Debt":>8} {"Age":>5} {"Approved?":>10}')
print('  ' + '-'*60)
for i, (app, label) in enumerate(zip(applicants, labels)):
    outcome = 'YES' if label == 1 else 'NO '
    print(f'  Applicant {i+1:<3} {app[0]:>8}k {app[1]:>8} {app[2]:>6}k {app[3]:>5}  {outcome}')

# What statistics alone would predict
approval_rate_high_income = 0.70
print(f'\nStatistics-only prediction: "High income → {approval_rate_high_income*100:.0f}% chance of approval"')
print('This gives the SAME prediction for ALL high-income applicants — even Applicant 5!')
print('\nApplicant 5: Income=$110k BUT credit=580 (poor) AND debt=$40k (high)')
print('→ Statistics says YES (70% chance), Real answer is NO')
print('→ A model using all 4 features would catch this.')

# Prediction error for statistics-only approach
stat_predictions = np.array([1, 1, 1, 1, 1, 1])  # always predict approved for high income
errors = np.abs(stat_predictions - labels)
print(f'\nStatistics-only prediction errors: {errors}')
print(f'Wrong on {errors.sum()} out of {len(labels)} applicants ({errors.mean()*100:.0f}% error rate)')

In [ ]:
# ── Visualization: Loan — Feature distributions for approved vs rejected ───────
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('Loan Approval: Approved vs Rejected Applicants Differ on Multiple Features',
             fontsize=12, fontweight='bold')

approved  = applicants[labels == 1]
rejected  = applicants[labels == 0]

for i, (ax, fname) in enumerate(zip(axes, feat_names)):
    positions = [1, 2]
    ax.bar([1], [approved[:, i].mean()], color='#55A868', alpha=0.8, width=0.4,
           label=f'Approved (n={len(approved)})')
    ax.bar([2], [rejected[:, i].mean()], color='#C44E52', alpha=0.8, width=0.4,
           label=f'Rejected (n={len(rejected)})')
    # scatter individual points
    ax.scatter(np.ones(len(approved))  * 1 + np.random.uniform(-0.1, 0.1, len(approved)),
               approved[:, i], color='#1a6e2e', zorder=5, s=50)
    ax.scatter(np.ones(len(rejected)) * 2 + np.random.uniform(-0.1, 0.1, len(rejected)),
               rejected[:, i], color='#8b0000', zorder=5, s=50)
    ax.set_title(fname)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Approved', 'Rejected'])
    if i == 0:
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()
print('Income alone does not separate approved from rejected — a model needs ALL features.')

### Key Insight

Income alone can't separate approvals from rejections when you look at the chart. All applicants have high incomes. What separates them is credit score and debt level — features that only a **model combining multiple inputs** can use.

---

## Scenario 2: Email Spam Detection

### What statistics tells us

An analysis of 10,000 emails shows:
- 80% of emails containing the word "FREE" are spam

### The limitation

Consider these two emails:
- Email A: "FREE iPhone! Click now! LIMITED OFFER!!!"
- Email B: "Free yourself from stress with our meditation app"

Both contain "FREE". Email A is clearly spam. Email B is marketing, but legitimate.

A single word is not enough. We need to look at **combination of signals**:
- How many ALL-CAPS words?
- How many exclamation marks?
- How many suspicious links?
- Is the sender domain known?

A model combines all these features into a spam score.

In [ ]:
# ── Email Spam: Feature profiles for several emails ───────────────────────────
# Features: [has_FREE, caps_word_count, exclamation_count, link_count, known_domain]
emails = {
    'FREE iPhone!!! CLICK NOW!!': [1, 4, 3, 2, 0],
    'Free yourself from stress':  [1, 0, 0, 0, 1],
    'Meeting at 3pm tomorrow':    [0, 0, 0, 0, 1],
    'FREE MONEY win prizes NOW!': [1, 5, 2, 3, 0],
    'Your order has shipped':     [0, 0, 0, 1, 1],
    'FREE trial - start today':   [1, 0, 0, 1, 1],
}
actual_spam = [1, 0, 0, 1, 0, 0]  # ground truth
feat_names  = ['Has FREE', 'CAPS Words', 'Exclamations', 'Links', 'Known Domain']

print('=== EMAIL FEATURE PROFILES ===')
header = f'{"Email (truncated)":40}  {"FREE":5}  {"CAPS":6}  {"!":4}  {"Links":6}  {"KnownDmn":9}  {"SPAM?":6}'
print(header)
print('-' * len(header))
for (email, feats), spam in zip(emails.items(), actual_spam):
    trunc = email[:38] + '..' if len(email) > 38 else email
    label = 'SPAM' if spam else 'NOT SPAM'
    print(f'{trunc:40}  {feats[0]:5}  {feats[1]:6}  {feats[2]:4}  {feats[3]:6}  {feats[4]:9}  {label}')

print('\nStatistics only: "Has FREE" → predict SPAM')
free_feature = [f[0] for f in emails.values()]
stat_preds   = free_feature  # using FREE-word as sole predictor
errors_spam  = np.abs(np.array(stat_preds) - np.array(actual_spam))
print(f'Errors using only FREE word: {errors_spam.sum()} / {len(actual_spam)} wrong')
print('\n"Free yourself from stress" and "FREE trial" would be falsely flagged!')

In [ ]:
# ── Visualization: Radar-style feature comparison for spam vs not-spam ─────────
email_list = list(emails.keys())
feat_arr   = np.array(list(emails.values()), dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Email Feature Profiles: Spam vs Not Spam', fontsize=13, fontweight='bold')

colors_email = ['#C44E52' if s else '#55A868' for s in actual_spam]
labels_email = [f'SPAM: {e[:25]}' if s else f'OK: {e[:25]}'
                for e, s in zip(email_list, actual_spam)]

x = np.arange(len(feat_names))
width = 0.13
ax = axes[0]
for i, (feats, color, label) in enumerate(zip(feat_arr, colors_email, labels_email)):
    ax.bar(x + i*width - 2.5*width, feats, width, color=color, alpha=0.7,
           label=label)
ax.set_title('All Email Feature Values')
ax.set_xticks(x)
ax.set_xticklabels(feat_names, rotation=15, ha='right')
ax.set_ylabel('Feature Value')
spam_patch   = mpatches.Patch(color='#C44E52', alpha=0.7, label='SPAM')
nospam_patch = mpatches.Patch(color='#55A868', alpha=0.7, label='NOT SPAM')
ax.legend(handles=[spam_patch, nospam_patch], loc='upper right')

# Heatmap view
ax2 = axes[1]
im  = ax2.imshow(feat_arr.T, aspect='auto', cmap='YlOrRd')
ax2.set_yticks(np.arange(len(feat_names)))
ax2.set_yticklabels(feat_names)
ax2.set_xticks(np.arange(len(email_list)))
ax2.set_xticklabels([f'E{i+1}\n{"SPAM" if s else "OK"}'
                     for i, s in enumerate(actual_spam)], rotation=15)
plt.colorbar(im, ax=ax2, label='Feature value')
ax2.set_title('Feature Heatmap (darker = higher value)')

# Annotate cells
for r in range(feat_arr.shape[1]):
    for c in range(feat_arr.shape[0]):
        ax2.text(c, r, f'{feat_arr[c, r]:.0f}', ha='center', va='center',
                 fontsize=10, color='black')

plt.tight_layout()
plt.show()

### Key Insight

The heatmap makes it clear: spam emails (E1, E4) have HIGH values across ALL features simultaneously. "Free yourself..." (E2) has FREE=1 but caps=0, exclamations=0, links=0 — it looks nothing like spam when you see all features together. The model needs all of them.

---

## Scenario 3: House Price Prediction

### What statistics tells us

City-wide data shows the average house price is $350,000.

### The limitation

A buyer asks: "What should I pay for this specific house — 3 beds, 2 baths, 1500 sqft, built 2010, good neighborhood?"

"$350k" is not a useful answer. A 5-bedroom new construction in a great neighborhood should be $600k. A studio apartment from 1970 should be $150k. The average hides all this variation.

**A model uses the specific features of the house to estimate its specific price.**

In [ ]:
# ── House Prices: Statistics vs model predictions ─────────────────────────────
# Features: [sqft, bedrooms, bathrooms, year_built, neighborhood_score]
houses = np.array([
    [1500, 3, 2, 2010, 8],
    [2800, 5, 3, 2018, 9],
    [ 900, 1, 1, 1975, 5],
    [2100, 4, 2, 2005, 7],
    [3200, 5, 4, 2020, 10],
    [1100, 2, 1, 1985, 4],
])
actual_prices = np.array([380, 580, 175, 420, 720, 210])  # in $k
avg_city_price = actual_prices.mean()  # the "statistic"

# Statistics-only prediction: always predict the mean
stat_pred_prices = np.full(len(actual_prices), avg_city_price)
stat_errors      = actual_prices - stat_pred_prices

print(f'=== HOUSE PRICE PREDICTION ===')
print(f'  City average price (the statistic): ${avg_city_price:.0f}k')
print(f'\n  {"House":8} {"Sqft":>6} {"Beds":>5} {"Year":>6} {"Score":>6} {"Actual":>8} {"Stat Pred":>10} {"Error":>8}')
print('  ' + '-' * 62)
for i, (h, actual, pred, err) in enumerate(
        zip(houses, actual_prices, stat_pred_prices, stat_errors)):
    print(f'  House {i+1:2}  {h[0]:>6} {h[1]:>5} {h[2]:>6}    {h[4]:>3}  '
          f'${actual:>5}k   ${pred:>6.0f}k   {err:>+6.0f}k')

# MSE for stats-only approach
mse_stat = (stat_errors ** 2).mean()
print(f'\n  MSE using only the average: {mse_stat:.0f} k² '
      f'(RMSE = ${np.sqrt(mse_stat):.0f}k off on average)')
print('  Every house gets the same prediction regardless of its actual features!')

In [ ]:
# ── Visualization: Actual prices vs statistic prediction ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('House Prices: The Statistic (Average) vs What We Need',
             fontsize=13, fontweight='bold')

house_labels = [f'H{i+1}' for i in range(len(actual_prices))]
x = np.arange(len(actual_prices))
width = 0.35

# Bar chart: actual vs stat prediction
ax = axes[0]
ax.bar(x - width/2, actual_prices, width, label='Actual Price', color='#4C72B0', alpha=0.8)
ax.bar(x + width/2, stat_pred_prices, width, label='Statistics prediction (mean)',
       color='#DD8452', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(house_labels)
ax.set_ylabel('Price ($k)')
ax.set_title('Actual vs Statistics-Only Prediction')
ax.legend()
ax.axhline(avg_city_price, color='gray', linewidth=0.8, linestyle=':', alpha=0.7)

# Error bars
ax2 = axes[1]
colors_err = ['#C44E52' if e > 0 else '#55A868' for e in stat_errors]
bars = ax2.bar(x, stat_errors, color=colors_err, alpha=0.8)
ax2.axhline(0, color='black', linewidth=1.5)
ax2.set_xticks(x)
ax2.set_xticklabels(house_labels)
ax2.set_ylabel('Error = Actual − Prediction ($k)')
ax2.set_title('Prediction Errors (using only the average)')
ax2.bar_label(bars, fmt='%+.0fk', padding=3)
ax2.text(0.02, 0.97, f'RMSE = ${np.sqrt(mse_stat):.0f}k',
         transform=ax2.transAxes, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

plt.tight_layout()
plt.show()

### Key Insight

The statistic (city average of ~$414k) is wrong for every single house — sometimes by $300k+ in either direction. A 1-bedroom from 1975 is not worth the same as a 5-bedroom built in 2020. A model that uses sqft, bedrooms, year, and neighborhood would produce dramatically smaller errors.

---

## Scenario 4: Student Grade Prediction

### What statistics tells us

Historical data shows the class average exam score is 72.

### The limitation

A new student is enrolled: studies 4 hours/day, attends 95% of classes, scored 85 on the last quiz. If you tell them "I predict you'll get 72" (the average), you are ignoring all their individual effort. They deserve a prediction based on their actual behavior.

Conversely, a student who studies 1 hour/day and attends 50% of classes should not be told "you'll get 72" either — that's too optimistic.

**Every student is different. The average tells us nothing about an individual.**

In [ ]:
# ── Student Grade Prediction ──────────────────────────────────────────────────
# Features: [study_hrs_per_day, attendance_pct, prev_quiz_score]
students = np.array([
    [4.0, 95, 85],   # diligent student
    [1.0, 50, 60],   # struggling student
    [2.5, 80, 72],   # average student
    [5.0, 99, 92],   # exceptional student
    [0.5, 40, 55],   # at-risk student
    [3.0, 85, 78],   # solid student
])
actual_grades = np.array([88, 58, 73, 95, 50, 79])
class_average = actual_grades.mean()

# Statistics-only prediction: use class average for everyone
stat_preds_g = np.full(len(actual_grades), class_average)
errors_g     = actual_grades - stat_preds_g

print(f'=== STUDENT GRADE PREDICTION ===')
print(f'  Class average (the statistic): {class_average:.1f}')
print(f'\n  {"Student":10} {"Study hrs":>10} {"Attend %":>9} {"Prev Quiz":>10}  '
      f'{"Actual":>7} {"Stat Pred":>10} {"Error":>8}')
print('  ' + '-' * 70)
profiles = ['Diligent', 'Struggling', 'Average', 'Exceptional', 'At-Risk', 'Solid']
for prof, s, actual, pred, err in zip(
        profiles, students, actual_grades, stat_preds_g, errors_g):
    print(f'  {prof:10}  {s[0]:>9.1f}  {s[1]:>8.0f}%  {s[2]:>9.0f}  '
          f'{actual:>7}  {pred:>9.1f}  {err:>+7.1f}')

mse_g = (errors_g ** 2).mean()
print(f'\n  MSE using class average: {mse_g:.1f}  (RMSE = {np.sqrt(mse_g):.1f} grade points off)')

In [ ]:
# ── Visualization: Study hours vs actual grade (showing relationship) ──────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Student Grade Prediction: Why the Class Average Fails Individuals',
             fontsize=13, fontweight='bold')

# Panel 1: study hours vs actual grade — the relationship the model learns
ax = axes[0]
scatter = ax.scatter(students[:, 0], actual_grades, s=100,
                     c=students[:, 1], cmap='RdYlGn',
                     vmin=30, vmax=100, zorder=5)
plt.colorbar(scatter, ax=ax, label='Attendance %')
ax.axhline(class_average, color='red', linewidth=2, linestyle='--',
           label=f'Class Average = {class_average:.0f}')
for prof, s, grade in zip(profiles, students, actual_grades):
    ax.annotate(prof, (s[0] + 0.05, grade), fontsize=8)
ax.set_xlabel('Study Hours per Day')
ax.set_ylabel('Actual Grade')
ax.set_title('Study Hours vs Grade\n(color = attendance)')
ax.legend()

# Panel 2: error of statistics-only approach
ax2 = axes[1]
colors_g = ['#C44E52' if abs(e) > 10 else '#DD8452' if abs(e) > 5 else '#55A868'
            for e in errors_g]
x_g = np.arange(len(profiles))
bars = ax2.bar(x_g, errors_g, color=colors_g, alpha=0.8)
ax2.axhline(0, color='black', linewidth=1.5)
ax2.set_xticks(x_g)
ax2.set_xticklabels(profiles, rotation=15, ha='right')
ax2.set_ylabel('Error = Actual − Mean Prediction')
ax2.set_title('Errors When Predicting Everyone Gets 72')
ax2.bar_label(bars, fmt='%+.0f', padding=3)

plt.tight_layout()
plt.show()

### Key Insight

Study hours and attendance are clearly correlated with grades (look at the scatter plot). The class average is wrong by up to 24 points for some students. A model that learns these relationships would give each student a meaningful, personalized prediction.

---

## Error Types and MSE: Measuring How Wrong We Are

### Concept

Every prediction has an error. The goal of machine learning is to minimize this error.

**Error for one prediction:**
$$\text{error}_i = y_i - \hat{y}_i$$

where $y_i$ is the actual value and $\hat{y}_i$ is our predicted value.

**Mean Squared Error (MSE)** — the standard way to measure overall prediction quality:
$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

We square errors because:
1. Negative and positive errors cancel out if we don't — squaring prevents that
2. Squaring penalizes large errors more than small ones (a $100 error is 100x worse than a $10 error, not just 10x)

In [ ]:
# ── MSE Step-by-Step Walkthrough ──────────────────────────────────────────────
# Concrete example: a house price model's predictions vs actuals
actuals    = np.array([380, 580, 175, 420, 720, 210])  # actual house prices $k
# Pretend a simple model outputs these predictions
model_preds = np.array([360, 610, 190, 400, 690, 230])  # model predictions $k

print('=== MSE CALCULATION — STEP BY STEP ===')
print(f'  {"Example":>8}  {"Actual":>8}  {"Predicted":>10}  {"Error":>8}  {"Error²":>10}')
print('  ' + '-' * 55)

individual_errors   = actuals - model_preds
squared_errors      = individual_errors ** 2

for i, (a, p, e, e2) in enumerate(
        zip(actuals, model_preds, individual_errors, squared_errors)):
    print(f'  House {i+1:>2}     ${a:>5}k     ${p:>6}k    {e:>+6}k    {e2:>8}k²')

print('  ' + '-' * 55)
print(f'  Sum of squared errors: {squared_errors.sum()}k²')
mse  = squared_errors.mean()
rmse = np.sqrt(mse)
print(f'  MSE  = {squared_errors.sum()} / {len(actuals)} = {mse:.1f}k²')
print(f'  RMSE = sqrt({mse:.1f}) = {rmse:.1f}k')
print(f'\n  Interpretation: on average, the model is ${rmse:.0f}k off per prediction')

# Compare: statistics-only vs model
stat_mse  = ((actuals - actuals.mean()) ** 2).mean()
stat_rmse = np.sqrt(stat_mse)
print(f'\n  Statistics-only RMSE (using mean): ${stat_rmse:.0f}k')
print(f'  Model RMSE                        : ${rmse:.0f}k')
print(f'  Model improvement                 : ${stat_rmse - rmse:.0f}k less error per prediction')

In [ ]:
# ── Visualization: Predictions vs Actuals ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Visualizing Prediction Errors', fontsize=13, fontweight='bold')

house_ids = [f'H{i+1}' for i in range(len(actuals))]
x = np.arange(len(actuals))
width = 0.35

# Bar chart: predictions vs actuals
ax = axes[0]
ax.bar(x - width/2, actuals,      width, label='Actual',    color='#4C72B0', alpha=0.85)
ax.bar(x + width/2, model_preds,  width, label='Predicted', color='#DD8452', alpha=0.85)
# Draw error lines
for xi, a, p in zip(x, actuals, model_preds):
    ymin, ymax = min(a, p), max(a, p)
    ax.plot([xi + width/2]*2, [ymin, ymax], color='red', linewidth=2, zorder=5)
ax.set_xticks(x)
ax.set_xticklabels(house_ids)
ax.set_ylabel('Price ($k)')
ax.set_title('Actual vs Predicted Prices\n(red lines = errors)')
ax.legend()

# Scatter: predicted vs actual with diagonal line (perfect predictions)
ax2 = axes[1]
ax2.scatter(actuals, model_preds, s=90, color='#4C72B0', zorder=5,
            label='Model predictions')
# Perfect prediction diagonal
diag_range = np.linspace(min(actuals) - 30, max(actuals) + 30, 100)
ax2.plot(diag_range, diag_range, 'k--', linewidth=1.5, alpha=0.7,
         label='Perfect prediction (y=x)')
# Error lines to diagonal
for a, p, name in zip(actuals, model_preds, house_ids):
    ax2.plot([a, a], [p, a], color='red', linewidth=1.5, alpha=0.7)
    ax2.annotate(name, (a + 5, p), fontsize=8)
ax2.set_xlabel('Actual Price ($k)')
ax2.set_ylabel('Predicted Price ($k)')
ax2.set_title('Scatter: Predicted vs Actual\n(closer to diagonal = better)')
ax2.legend()
ax2.text(0.02, 0.97, f'RMSE = ${rmse:.0f}k', transform=ax2.transAxes,
         verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

### Key Insight

The diagonal line in the scatter plot is your target. Points ON the diagonal = perfect predictions, error = 0. Points ABOVE the diagonal = model underpredicted. Points BELOW = model overpredicted. The goal of training a model is to move all points as close to the diagonal as possible — which means minimizing the MSE.

---

## Supervised Learning: How Models Are Trained

### Concept

**Supervised learning** means the model learns from examples where we already know the correct answer.

```
Training data:  [features₁, actual_output₁]
                [features₂, actual_output₂]
                     ...many examples...
```

The model:
1. Looks at the features
2. Makes a prediction
3. Compares to the actual answer — this is the **error**
4. Adjusts its internal settings to reduce that error
5. Repeats thousands of times

Eventually, the model's internal settings encode the true relationships in the data.

In [ ]:
# ── Visualize the supervised learning loop (conceptual diagram) ──────────────
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 5)
ax.axis('off')
ax.set_title('The Supervised Learning Loop', fontsize=14, fontweight='bold', pad=15)

# Draw boxes
box_specs = [
    (1.0, 3.0, 1.8, 1.2, '#AED6F1', 'Training Data\n(features + labels)'),
    (3.5, 3.0, 1.8, 1.2, '#A9DFBF', 'Model\n(adjustable weights)'),
    (6.0, 3.0, 1.8, 1.2, '#FAD7A0', 'Prediction\nŷ'),
    (6.0, 1.0, 1.8, 1.2, '#F1948A', 'Error\n(ŷ - y)'),
    (3.5, 1.0, 1.8, 1.2, '#D7BDE2', 'Weight Update\n(reduce error)'),
]
for (bx, by, bw, bh, color, label) in box_specs:
    rect = mpatches.FancyBboxPatch((bx, by), bw, bh,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='gray', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(bx + bw/2, by + bh/2, label, ha='center', va='center',
            fontsize=10, fontweight='bold')

# Arrows
arrow_props = dict(arrowstyle='->', color='#333333', lw=2.0)
arrows = [
    ((2.8, 3.6), (3.5, 3.6)),  # Data -> Model
    ((5.3, 3.6), (6.0, 3.6)),  # Model -> Prediction
    ((6.9, 3.0), (6.9, 2.2)),  # Prediction -> Error
    ((6.0, 1.6), (5.3, 1.6)),  # Error -> Weight Update
    ((3.5, 1.6), (4.4, 3.0)),  # Weight Update -> Model (feedback loop)
]
for (start, end) in arrows:
    ax.annotate('', xy=end, xytext=start,
                arrowprops=arrow_props)

# Labels on arrows
ax.text(3.1, 3.85, 'features', fontsize=9, color='gray', style='italic')
ax.text(5.4, 3.85, 'makes prediction', fontsize=9, color='gray', style='italic')
ax.text(7.1, 2.6,  'compare to\nactual y', fontsize=9, color='gray', style='italic')
ax.text(5.5, 1.85, 'signal', fontsize=9, color='gray', style='italic')
ax.text(3.6, 2.3,  'feedback\nloop', fontsize=9, color='gray', style='italic')

# Actual value box (feeding into error)
rect2 = mpatches.FancyBboxPatch((7.8, 1.0), 1.8, 1.2,
                                  boxstyle='round,pad=0.1',
                                  facecolor='#FDFEFE', edgecolor='gray', linewidth=1.0,
                                  linestyle='dashed')
ax.add_patch(rect2)
ax.text(8.7, 1.6, 'Actual y\n(ground truth)', ha='center', va='center', fontsize=9)
ax.annotate('', xy=(7.8, 1.6), xytext=(7.8, 1.6),  # placeholder
            arrowprops=arrow_props)
ax.annotate('', xy=(7.8, 1.6), xytext=(8.0, 1.6),
            arrowprops=dict(arrowstyle='<-', color='#333333', lw=2.0))

plt.tight_layout()
plt.show()

---

## Overfitting: When a Model Memorizes Instead of Learning

### Concept

Imagine a student who memorizes every answer to last year's exam word-for-word. They score perfectly on that exam — but fail the new exam because they didn't actually understand the material.

This is **overfitting** in machine learning:
- The model learns the training data so perfectly (including its noise and quirks)
- that it fails on new, unseen data

### Train vs Test Split

To detect overfitting, we always split our data:

| Split | Used for | Size |
|-------|----------|------|
| Training set | Teaching the model | 70-80% |
| Test set | Evaluating the model on NEW data | 20-30% |

A good model has LOW error on BOTH training and test data.  
An overfit model has LOW training error but HIGH test error.

In [ ]:
# ── Overfitting Illustration (conceptual) ─────────────────────────────────────
np.random.seed(7)

# True underlying relationship: grade = 60 + 8 * study_hours
study_h_all = np.linspace(0.5, 5.5, 30)
true_grades = 60 + 8 * study_h_all
noisy_grades = true_grades + np.random.normal(0, 6, 30)

# Split 70/30 train/test
n_train = int(0.7 * len(study_h_all))
study_train, study_test   = study_h_all[:n_train],   study_h_all[n_train:]
grade_train, grade_test   = noisy_grades[:n_train],  noisy_grades[n_train:]

# Simple model (good generalization) — linear
coeffs_simple = np.polyfit(study_train, grade_train, deg=1)
pred_train_simple = np.polyval(coeffs_simple, study_train)
pred_test_simple  = np.polyval(coeffs_simple, study_test)

# Complex overfit model — high degree polynomial fitted to training data
coeffs_complex = np.polyfit(study_train, grade_train, deg=10)
x_smooth = np.linspace(0.5, 5.5, 300)
pred_complex_smooth   = np.polyval(coeffs_complex, x_smooth)
pred_train_complex    = np.polyval(coeffs_complex, study_train)
pred_test_complex     = np.polyval(coeffs_complex, study_test)

mse_train_simple  = ((grade_train - pred_train_simple)  ** 2).mean()
mse_test_simple   = ((grade_test  - pred_test_simple)   ** 2).mean()
mse_train_complex = ((grade_train - pred_train_complex) ** 2).mean()
mse_test_complex  = ((grade_test  - pred_test_complex)  ** 2).mean()

print('=== OVERFITTING COMPARISON ===')
print(f'  {"":25}  {"Train MSE":>10}  {"Test MSE":>10}')
print(f'  {"Simple (linear) model":25}  {mse_train_simple:>10.1f}  {mse_test_simple:>10.1f}')
print(f'  {"Complex (overfit) model":25}  {mse_train_complex:>10.1f}  {mse_test_complex:>10.1f}')
print(f'\n  The overfit model has lower training error but MUCH higher test error!')
print(f'  It memorized the training noise, not the real pattern.')

In [ ]:
# ── Visualization: Good model vs overfit model ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Overfitting: Memorizing Training Data vs Learning the True Pattern',
             fontsize=13, fontweight='bold')

x_plot = np.linspace(0.5, 5.5, 300)

for ax, (model_name, train_mse, test_mse, y_pred_smooth) in zip(axes, [
    ('Simple (Linear) Model — Good Generalization',
     mse_train_simple, mse_test_simple,
     np.polyval(coeffs_simple, x_plot)),
    ('Complex (Degree-10) Model — Overfit',
     mse_train_complex, mse_test_complex,
     np.clip(np.polyval(coeffs_complex, x_plot), 40, 115)),
]):
    ax.scatter(study_train, grade_train, color='#4C72B0', s=60, zorder=5, label='Training data')
    ax.scatter(study_test,  grade_test,  color='#C44E52', s=60, marker='D', zorder=5, label='Test data')
    ax.plot(x_plot, y_pred_smooth, color='#2CA02C', linewidth=2.5, label='Model')
    ax.set_title(f'{model_name}\nTrain MSE={train_mse:.1f} | Test MSE={test_mse:.1f}',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Study Hours per Day')
    ax.set_ylabel('Grade')
    ax.set_ylim(45, 115)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
print('The overfit model bends wildly to fit every training point,'
      ' then makes poor predictions on test data.')

### Key Insight

The simple linear model doesn't perfectly fit every training point — but it captures the real pattern. The complex model perfectly follows every training point (including noise), but when it sees the test data (red diamonds), it is badly wrong. **Low training error is NOT the goal. Low test error is the goal.**

---

## Week 2 Checkpoint

After completing this notebook you should be able to:

1. **Explain why statistics are insufficient for prediction**: a group statistic (like a 70% approval rate or a $350k average price) gives the same answer for everyone in the group, while a model uses each individual's specific feature values to produce a personalized prediction.

2. **Compute prediction error** step by step — subtract each prediction from its actual value, square the differences, take the mean — and interpret RMSE as the average dollar/grade/unit the model is off by.

3. **Read a predicted-vs-actual scatter plot**: points on the diagonal = perfect predictions; the distance from the diagonal shows error magnitude; points above = underpredicted; points below = overpredicted.

4. **Explain overfitting** using the train/test framework: a model with low training error but high test error has memorized training noise rather than learned the true pattern, and the fix is to evaluate on data the model has never seen.